In [3]:
import keepa
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

True

## Data Generation Process

To generate training data, we constructed two complementary tables: a **Product table** and a **Prices table**, using 1,000 electronics ASINs sourced from the Keepa bestseller list. The Product table contains relatively static metadata and baseline statistics for each product, including ASIN, category, brand, global minimum price, global maximum price, and global standard deviation (over the last 3 years). These features were selected to capture a product’s overall pricing behavior and reputation in the market. The minimum and maximum prices define the natural range of the product, while the standard deviation reflects how volatile its price is over time. Together, these statistics help the model understand whether a product typically has stable pricing or fluctuates frequently, which directly impacts how predictable its future prices are.

The Prices table captures time-dependent and market-driven features, including ASIN, datetime, month, day of week, brand, Amazon price, lowest new marketplace price, lowest used marketplace price, list price, number of new sellers, number of used sellers, and a transformed sales score defined as −log(sales rank). Month and day of week were included to allow the model to identify temporal patterns in pricing, such as discounts during major shopping periods like Black Friday or Christmas. The marketplace features were chosen to explicitly model competition: if another seller offers a lower new price, Amazon is more likely to decrease its price to remain competitive, whereas if Amazon faces little competition such as in the used market they has more flexibility to increase prices with little consequence. The number of sellers reinforces this dynamic by indicating supply levels. The sales score is essential because it reflects demand in a more linear way, allowing the model to understand that highly ranked (popular) products can tolerate price changes with less impact on sales, while less popular products are more price-sensitive.

In addition, we engineered several price-based features to capture short-term trends, including lagged prices, moving averages, price deltas, and percentage price changes over 7- and 14-day intervals. These features help the model detect momentum and recent pricing behavior, allowing it to understand not just the current price but how it has been evolving over time in both weekly and biweekly windows. For data collection, we queried the past three years of price history for each product, storing the least recent two years at a weekly frequency and the most recent year at a daily frequency to balance detail with efficiency. After processing the initial 1,000 products, we determined that 310 had sufficient Amazon data to be usable, resulting in a final Prices table containing 21,206 rows.

In [4]:
accesskey = os.getenv('KEEPA_API_KEY')
api = keepa.Keepa(accesskey)

In [5]:
# 1) Get 1000 Electronics ASINs
asins_csv = pd.read_csv("KeepaAsins.csv")
electronics_asins = asins_csv['ASIN'].tolist()

print(f"Collected {len(electronics_asins)} Electronics ASINs")
print(electronics_asins[:10])

Collected 1000 Electronics ASINs
['B0D7FVQ1ZB', 'B0DGHMNQ5Z', 'B0GJTFXNRX', 'B0DZ7871B8', 'B0FQFB8FMG', 'B09PDLBFKY', 'B0DXXYS4BJ', 'B08R6S1M1K', 'B0D6GFNFJY', 'B0BQS9Y69G']


In [6]:
# 2) Helpers
KEEPA_EPOCH = datetime.datetime(2011, 1, 1) if "datetime" in globals() and hasattr(datetime, "datetime") else datetime(2011, 1, 1)

def keepa_to_datetime(keepa_minutes):
    return KEEPA_EPOCH + timedelta(minutes=int(keepa_minutes))

# product table: asin, category, sales rank, ratings, review count, brand, global min/max, global sd (amazon)
# prices table: asin, (rest of stuff below), LISTPRICE, USED, COUNT_USED

CSV_IDX = {
    "AMAZON": 0,
    "NEW": 1,
    "USED": 2,
    "SALES": 3,
    "LISTPRICE": 4,
    "COUNT_NEW": 11,
    "COUNT_USED": 12,
}

PRICE_SERIES = {"AMAZON", "NEW", "USED", "LISTPRICE"}

def get_csv_series(product, idx):
    csv_data = product.get("csv", None)
    if csv_data is None or idx >= len(csv_data):
        return None
    return csv_data[idx]

def parse_pair_series(series, name, convert_price=False):
    if not series:
        return pd.DataFrame(columns=["keepa_minutes", "datetime", name])

    rows = []
    for i in range(0, len(series), 2):
        if i + 1 >= len(series):
            break
        t = series[i]
        v = series[i + 1]

        if v == -1:
            parsed = None
        else:
            parsed = v / 100.0 if convert_price else v

        rows.append({
            "keepa_minutes": t,
            "datetime": keepa_to_datetime(t),
            name: parsed
        })

    return pd.DataFrame(rows)

def downsample_first_two_years_weekly_last_year_daily(df, price_cols):
    """First 2 years weekly, last year daily."""
    if df.empty:
        return df

    df = df.sort_values("datetime").copy()
    end_date = df["datetime"].max()
    start_3yrs = end_date - pd.DateOffset(years=3)
    cutoff_2yrs = start_3yrs + pd.DateOffset(years=2)

    # Trim to exactly 3 years
    df = df[df["datetime"] >= start_3yrs]

    # Split
    first_2yrs = df[df["datetime"] < cutoff_2yrs]
    last_year = df[df["datetime"] >= cutoff_2yrs]

    # --- First 2 years: WEEKLY ---
    if not first_2yrs.empty:
        first_2yrs = first_2yrs.set_index("datetime")
        first_2yrs = first_2yrs.resample("7D").first().reset_index()

        for col in df.columns:
            if col not in price_cols + ["datetime"]:
                first_2yrs[col] = first_2yrs[col].ffill()

    # --- Last year: DAILY ---
    if not last_year.empty:
        last_year = last_year.set_index("datetime")
        last_year = last_year.resample("1D").first().reset_index()

        for col in df.columns:
            if col not in price_cols + ["datetime"]:
                last_year[col] = last_year[col].ffill()

    combined = pd.concat([first_2yrs, last_year], ignore_index=True)
    return combined.sort_values("datetime").reset_index(drop=True)

def build_aligned_keepa_df(product):
    raw_cols = list(CSV_IDX.keys())
    dfs = []

    product_type = product.get("productType", 0)
    if product_type in {3, 4, 5}:  # inaccessible, invalid, variation parent
        return pd.DataFrame(columns=[
            "asin", "brand", "keepa_minutes", "datetime",
            "AMAZON", "NEW", "COUNT_NEW", "SALES_SCORE"
        ])

    for name in raw_cols:
        idx = CSV_IDX[name]
        series = get_csv_series(product, idx)

        df = parse_pair_series(
            series,
            name=name,
            convert_price=(name in PRICE_SERIES)
        )
        dfs.append(df)

    timeline_sources = [df[["keepa_minutes", "datetime"]] for df in dfs if not df.empty]
    if not timeline_sources:
        return pd.DataFrame(columns=[
            "asin", "brand", "keepa_minutes", "datetime",
            "AMAZON", "NEW", "COUNT_NEW", "SALES_SCORE"
        ])

    timeline = (
        pd.concat(timeline_sources, ignore_index=True)
        .drop_duplicates()
        .sort_values("keepa_minutes")
        .reset_index(drop=True)
    )

    out = timeline.copy()

    for df in dfs:
        if not df.empty:
            out = out.merge(df, on=["keepa_minutes", "datetime"], how="left")

    for col in raw_cols:
        if col not in out.columns:
            out[col] = pd.NA

    # Only forward fill NON-AMAZON columns
    cols_to_fill = [c for c in raw_cols if c != "AMAZON"]
    out[cols_to_fill] = out[cols_to_fill].ffill()

    # fallback: if NEW is missing, use AMAZON
    out["NEW"] = out["NEW"].fillna(out["AMAZON"])

    # convert sales rank to a model-friendlier score
    out["SALES_RANK"] = out["SALES"]
    out["SALES_SCORE"] = -np.log1p(out["SALES"])

    # Seasonality features
    out["DAY_OF_WEEK"] = out["datetime"].dt.dayofweek
    out["MONTH"] = out["datetime"].dt.month

    out.insert(0, "asin", product.get("asin"))
    out.insert(1, "brand", product.get("brand"))

    # Downsample first 2 years weekly, enforce exact last 3 years
    out = downsample_first_two_years_weekly_last_year_daily(out, price_cols=["AMAZON", "NEW"])

    out = out.sort_values("datetime")

    # Keep only real AMAZON observations BEFORE features
    out = out[out["AMAZON"].notna()]

    if out.empty:
        return pd.DataFrame()

    # --- LAGS ---
    out["AMAZON_LAG_7D"] = out.groupby("asin")["AMAZON"].shift(7)
    out["AMAZON_LAG_14D"] = out.groupby("asin")["AMAZON"].shift(14)

    # --- MOVING AVERAGES ---
    out["AMAZON_MA_7D"] = (
        out.groupby("asin")["AMAZON"]
        .rolling(7, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    out["AMAZON_MA_14D"] = (
        out.groupby("asin")["AMAZON"]
        .rolling(14, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # --- DERIVED FEATURES ---
    out["AMAZON_DELTA_7D"] = out["AMAZON"] - out["AMAZON_LAG_7D"]
    out["AMAZON_PCT_CHANGE_7D"] = out["AMAZON"] / out["AMAZON_LAG_7D"] - 1

    out = out.drop(columns=["SALES", "SALES_RANK"])

    return out

In [7]:
# 3) Query the 1000 ASINs in batches
all_dfs = []
all_products = []
batch_size = 25

num_batches = (len(electronics_asins) + batch_size - 1) // batch_size

for i in range(num_batches):
    batch_asins = electronics_asins[i * batch_size:(i + 1) * batch_size]
    print(f"Querying batch {i + 1} / {num_batches}")

    try:
        products = api.query(batch_asins, history=True)
        all_products.extend(products)

        for product in products:
            try:
                df = build_aligned_keepa_df(product)
                if not df.empty:
                    all_dfs.append(df)
            except Exception as e:
                print(f"Skipped ASIN {product.get('asin')} due to parsing error: {e}")

    except Exception as e:
        print(f"Batch failed for ASINs {batch_asins[:3]}... : {e}")

Querying batch 1 / 40


100%|██████████| 25/25 [00:03<00:00,  6.85it/s]
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])


Querying batch 2 / 40


100%|██████████| 25/25 [00:05<00:00,  4.42it/s]
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

Querying batch 3 / 40


...

100%|██████████| 25/25 [00:04<00:00,  5.63it/s]
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

Querying batch 39 / 40


100%|██████████| 25/25 [00:05<00:00,  4.29it/s]
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

Querying batch 40 / 40


100%|██████████| 25/25 [00:03<00:00,  7.09it/s]
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["NEW"] = out["NEW"].fillna(out["AMAZON"])
C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\372450525.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

In [8]:
product_rows = []

for product in all_products:
    try:
        asin = product.get("asin")

        amazon_series = get_csv_series(product, CSV_IDX["AMAZON"])
        amazon_vals = [v for i, v in enumerate(amazon_series or []) if i % 2 == 1 and v != -1]

        if not amazon_vals:
            continue

        amazon_vals = np.array(amazon_vals) / 100.0

        product_rows.append({
            "asin": asin,
            "category": product.get("rootCategory"),
            "brand": product.get("brand"),
            "global_min": np.min(amazon_vals),
            "global_max": np.max(amazon_vals),
            "global_sd": np.std(amazon_vals),
        })

    except Exception as e:
        print(f"Product table skip {asin}: {e}")

product_df = pd.DataFrame(product_rows)

In [9]:
# 5) Combine and export to CSV
if all_dfs:
    prices_df = pd.concat(all_dfs, ignore_index=True)
    prices_df = prices_df.sort_values(["asin", "datetime"]).reset_index(drop=True)

    # Ensure only ASINs with amazon price exist
    valid_asins = prices_df.groupby("asin")["AMAZON"].apply(lambda x: x.notna().any())
    valid_asins = valid_asins[valid_asins].index

    prices_df = prices_df[prices_df["asin"].isin(valid_asins)]

    product_df = product_df[product_df["asin"].isin(valid_asins)]

    prices_df.to_csv("Prices.csv", index=False)
    product_df.to_csv("Product.csv", index=False)

    print("Prices shape:", prices_df.shape)
    print("Products shape:", product_df.shape)
else:
    print("No data collected.")

C:\Users\rjajo\AppData\Local\Temp\ipykernel_36232\3224967792.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prices_df = pd.concat(all_dfs, ignore_index=True)


Prices shape: (21240, 19)
Products shape: (312, 6)
